# HR.TRG_DEP INSERT Conversion

**Conversion Timestamp:** 2024-07-30 12:00:00

This notebook converts an ODI SQL `INSERT` statement targeting `HR.TRG_DEP` from `HR.DEPARTMENTS` into Databricks Spark SQL.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process WID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "Last Extract Time (YYYY-MM-DD HH:MM:SS)")
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "2999-12-31 23:59:59", "Current Extract Time (YYYY-MM-DD HH:MM:SS)")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Staging Table

No explicit C$ staging table operations were found in the source for this step.

## Flow Table

No explicit I$ flow table operations were found in the source for this step.

## Error / Check Tables

No explicit E$ error table or `SNP_CHECK_TAB` operations were found in the source for this step.

## PK Violation Detection

No explicit PK violation detection logic was found in the source for this step.

## Mark Records for Update

No explicit update flagging logic (`IND_UPDATE`) was found in the source for this step.

## MERGE into Target

In [ ]:
%sql
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}

INSERT INTO workspace.hr.trg_dep
  (
    department_id ,
    department_name ,
    manager_id ,
    location_id 
  ) 
SELECT 
  departments.department_id ,
  departments.department_name ,
  departments.manager_id ,
  departments.location_id  
FROM 
  workspace.hr.departments AS departments;

## Optimize Target

In [ ]:
%sql
-- Disable ZORDER stats check to prevent DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.hr.trg_dep ZORDER BY (department_id);

## Cleanup

No temporary tables (C$, I$) were explicitly created in this specific conversion snippet, so no cleanup is needed.

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_rows FROM workspace.hr.trg_dep;

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_dep LIMIT 10;

## Conversion Notes

*   The Oracle hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable to Delta Lake and Spark SQL.
*   All schema and table names have been converted to lowercase and prefixed with `workspace.` (e.g., `HR.TRG_DEP` -> `workspace.hr.trg_dep`).
*   Standard Databricks widgets have been included, although not directly used by this specific `INSERT` statement. This adheres to the standard notebook structure.
*   An `OPTIMIZE ... ZORDER BY` statement has been added with `department_id` as a ZORDER key, preceded by the necessary `SET` command to prevent `DELTA_ZORDERING_ON_COLUMN_WITHOUT_STATS` errors.